# 11. Carbon Cycle Tipping Points
## Critical Thresholds and Irreversibility

This notebook reproduces Figure 11 from the CARBONICA paper: Analysis of carbon cycle tipping points.

In [ ]:
# Import CARBONICA
import sys
sys.path.append('..')
from carbonica import CARBONICA
from carbonica.modules.ocean_sink import OceanSinkModel
from carbonica.modules.permafrost_engine import PermafrostEngine
from carbonica.modules.quantum_yield_tracker import QuantumYieldTracker

# Initialize
carbonica = CARBONICA(data_dir="../data")
ocean = OceanSinkModel()
perma = PermafrostEngine()
qyt = QuantumYieldTracker()

In [ ]:
# CARBONICA tipping thresholds
print("=== CARBONICA Tipping Thresholds (2025 Status) ===\n")

thresholds = [
    ("Amazon Forest Dieback", 
     "Amazon Φ_q < 0.042",
     qyt.get_quantum_yield(2025, 'amazon_tropical'),
     0.042,
     "2033-2041"),
    
    ("North Atlantic Sink Collapse",
     "Revelle Factor ≥ 13.5",
     ocean.get_revelle_factor(2025),
     13.5,
     "2044-2058"),
    
    ("Permafrost Self-Sustaining",
     "F_perma ≥ 2.8 PgC/yr",
     perma.get_flux(2025),
     2.8,
     "2038-2046"),
    
    ("Global Soil Carbon Release",
     "τ_soil < 18 years",
     carbonica.get_parameter('tau_soil', 2025),
     18,
     "2065-2085")
]

for name, condition, current, threshold, projection in thresholds:
    status = "🔴 CROSSED" if current >= threshold else "🟡 APPROACHING"
    if name == "Amazon Forest Dieback":
        status = "🔴 CROSSED" if current <= threshold else "🟡 APPROACHING"
    print(f"{name}:")
    print(f"  {condition}")
    print(f"  Current: {current:.3f} (threshold: {threshold})")
    print(f"  Status: {status}")
    print(f"  Projected crossing: {projection}\n")

In [ ]:
# Amazon dieback analysis
print("=== Amazon Forest Dieback ===")
phi_current = qyt.get_quantum_yield(2025, 'amazon_tropical')
phi_critical = 0.042
phi_gap = phi_current - phi_critical
phi_decline_rate = 0.021 / 10  # 2.1% per decade = 0.0021 per year

years_to_cross = phi_gap / phi_decline_rate
print(f"Current Φ_q: {phi_current:.3f}")
print(f"Critical Φ_q: {phi_critical:.3f}")
print(f"Gap: {phi_gap:.3f}")
print(f"Decline rate: {phi_decline_rate:.4f} per year")
print(f"Years to critical: {years_to_cross:.0f} years")
print(f"Crossing year: {2025 + years_to_cross:.0f}")
print(f"Paper range: 2033-2041")

In [ ]:
# North Atlantic sink collapse
print("\n=== North Atlantic Sink Collapse ===")
R_current = ocean.get_revelle_factor(2025)
R_critical = 13.5
R_gap = R_critical - R_current
R_accel = 0.067  # per year from paper

years_to_cross = R_gap / R_accel
print(f"Current R: {R_current:.1f}")
print(f"Critical R: {R_critical:.1f}")
print(f"Gap: {R_gap:.1f}")
print(f"Acceleration rate: {R_accel:.3f} per year")
print(f"Years to critical: {years_to_cross:.0f} years")
print(f"Crossing year: {2025 + years_to_cross:.0f}")
print(f"Paper range: 2044-2058")

In [ ]:
# Permafrost self-sustaining threshold
print("\n=== Permafrost Self-Sustaining ===")
F_current = perma.get_flux(2025)
F_critical = 2.8
F_gap = F_critical - F_current
F_accel = 0.12  # PgC/yr² from paper

# Quadratic: F = F0 + a*t + 0.5*b*t²
# Solve for t when F = F_critical
import math
a = 0.12  # linear term
b = 0.02  # quadratic term (estimated)

# Simplified linear estimate
years_linear = F_gap / a
print(f"Current F_perma: {F_current:.2f} PgC/yr")
print(f"Critical F_perma: {F_critical:.1f} PgC/yr")
print(f"Gap: {F_gap:.2f} PgC/yr")
print(f"Acceleration rate: {a:.2f} PgC/yr²")
print(f"Linear estimate: {years_linear:.0f} years → {2025 + years_linear:.0f}")
print(f"Paper range: 2038-2046")

In [ ]:
# Global soil carbon release
print("\n=== Global Soil Carbon Release ===")
tau_current = carbonica.get_parameter('tau_soil', 2025)
tau_critical = 18
tau_gap = tau_current - tau_critical
tau_decline = 0.2  # years per year (estimated)

years_to_cross = tau_gap / tau_decline
print(f"Current τ_soil: {tau_current} years")
print(f"Critical τ_soil: {tau_critical} years")
print(f"Gap: {tau_gap} years")
print(f"Decline rate: {tau_decline:.1f} years/year")
print(f"Years to critical: {years_to_cross:.0f} years")
print(f"Crossing year: {2025 + years_to_cross:.0f}")
print(f"Paper range: 2065-2085")

In [ ]:
# PCSI critical threshold
print("\n=== PCSI Critical Threshold ===")
pcsi_current = carbonica.compute_pcsi(2025)
pcsi_critical = 1.0
pcsi_gap = pcsi_critical - pcsi_current
pcsi_accel = 0.012  # per year from paper

years_to_cross = pcsi_gap / pcsi_accel
print(f"Current PCSI: {pcsi_current:.2f}")
print(f"Critical PCSI: {pcsi_critical:.1f}")
print(f"Gap: {pcsi_gap:.2f}")
print(f"Acceleration rate: {pcsi_accel:.3f} per year")
print(f"Years to critical: {years_to_cross:.0f} years")
print(f"Crossing year: {2025 + years_to_cross:.0f}")
print(f"Paper range (SSP3-7.0): 2047-2053")

In [ ]:
# Summary table
print("\n=== Tipping Point Summary ===")
print("-" * 60)
print(f"{'Tipping Point':<30} {'Current':<10} {'Threshold':<10} {'Crossing'}")
print("-" * 60)
print(f"{'Amazon Dieback':<30} {phi_current:<10.3f} {phi_critical:<10.3f} 2033-2041")
print(f"{'N Atlantic Collapse':<30} {R_current:<10.1f} {R_critical:<10.1f} 2044-2058")
print(f"{'Permafrost Self-sustaining':<30} {F_current:<10.2f} {F_critical:<10.1f} 2038-2046")
print(f"{'Soil Carbon Release':<30} {tau_current:<10.0f} {tau_critical:<10.0f} 2065-2085")
print(f"{'PCSI Saturation':<30} {pcsi_current:<10.2f} {pcsi_critical:<10.1f} 2047-2053")
print("-" * 60)